In [ ]:
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInRead", "CORRECTED")
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")
spark.conf.set("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED")
spark.conf.set("spark.sql.parquet.int96RebaseModeInWrite", "CORRECTED")


In [ ]:
# Improved, more readable and optimized version without using read_table_filtered
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from datetime import datetime, timedelta
from sentinel_lake.providers import MicrosoftSentinelProvider, CustomDataFrame
import time

LOG_ANALYTICS_WORKSPACE = "<YOUR_WORKSPACE_NAME>"

# Provider instance (imported elsewhere in the notebook)
lake_provider = MicrosoftSentinelProvider(spark=spark)

# Utility function for logging
def cache_and_log_step(step_name, df, start_time=time.time()) -> DataFrame:
    """Log step execution with count and timing"""
    infilecnt = df.inputFiles().__len__()
    dfc = df.cache()
    cache_duration = time.time() - start_time
    start_time = time.time()
    count = dfc.count()
    count_duration = time.time() - start_time
    print(f"✓ {step_name}: {count:,} rows from {infilecnt} files (Cached in {cache_duration:.2f}s, Counted in {count_duration:.2f}s)")
    if isinstance(dfc, CustomDataFrame):
        return dfc.df
    return dfc

# Common time filter
time_filter = (F.col('TimeGenerated') >= F.expr("current_timestamp() - INTERVAL 7 DAYS"))
ti_time_filter = (F.col('TimeGenerated') >= F.expr("current_timestamp() - INTERVAL 14 DAYS"))


### Overrides for testing. Remove before giving back to customer


## Start of analytics

In [ ]:

print("=" * 80)
print("DATA LOADING AND ENRICHMENT PIPELINE")
print("=" * 80)
 
# EmailEvents (only inbound)
start = time.time()
df_emails = (
    lake_provider.read_table("EmailEvents", LOG_ANALYTICS_WORKSPACE)
    .filter(time_filter & (F.col("EmailDirection") == "Inbound"))
    .select(
        "TimeGenerated",
        "NetworkMessageId",
        "SenderFromAddress",
        "RecipientEmailAddress",
        "Subject",
        F.col("EmailClusterId").cast("string").alias("EmailClusterId")
    )
)
df_emails = cache_and_log_step("1. df_emails (Inbound emails)", df_emails, start)

In [ ]:
 # EmailUrlInfo
start = time.time()
df_urls = (
    lake_provider.read_table("EmailUrlInfo", LOG_ANALYTICS_WORKSPACE)
    .filter(time_filter)
    .select(
        "TimeGenerated",
        F.col("UrlDomain").alias("Domain"),
        "Url",
        "NetworkMessageId"
    )
)
df_urls = cache_and_log_step("2. df_urls (Email URLs)", df_urls, start)

In [ ]:
# EmailAttachmentInfo (exclude common image extensions, case-insensitive)
image_extensions = r".*\.(jpeg|jpg|png|gif|bmp|tiff|tif|webp|svg|ico|heic|heif|raw|psd|ai|eps)$"
has_extension = r".*\.[a-zA-Z0-9]{1,}$"
 
start = time.time()
df_attachments = (
    lake_provider.read_table("EmailAttachmentInfo", LOG_ANALYTICS_WORKSPACE)
    .filter(time_filter)
    #.filter(~F.lower(F.col("FileName")).rlike(image_extensions))  # Exclude image files
    #.filter(F.col("FileName").rlike(has_extension))  # Keep only files WITH extension
    .select(
        "TimeGenerated",
        "NetworkMessageId",
        "FileName",
        "SHA256"
    )
)
df_attachments = cache_and_log_step("3. df_attachments (Non-image files with extension)", df_attachments, start)


### attachment_identity
removed mapping to email

In [ ]:
# Attachment identity: distinct attachments with SHA256 
# Removed per email attachment
start = time.time()
attachment_identity = (
    df_attachments
    .select(
        F.lower(F.col("FileName")).alias("filename_lower"),
        "FileName",
        "SHA256"
    )
    .where(F.col("SHA256").isNotNull())
    .distinct()
)
attachment_identity = cache_and_log_step("4. attachment_identity (Distinct attachments with SHA256)", attachment_identity, start)

## 5. Device File Events 


### Updated: Single join, no collect, let Spark decide broadcast, only SHA256 in lookup, no networkmessageid in drop_duplicates

In [ ]:
# Optimized: Single join, no collect(), let Spark decide broadcast, only SHA256 in lookup, no networkmessageid in drop_duplicates
start = time.time()

# Prepare the attachment lookup table
attachment_lookup = (
    attachment_identity
    .select("SHA256")
    .distinct()
    .cache()
)
print(f"  → {attachment_lookup.count():,} distinct attachment SHA256 values to match")

# TODO: This is quite narrow - consider broadening to other email clients/processes
df_device_files = (
    lake_provider.read_table("DeviceFileEvents", LOG_ANALYTICS_WORKSPACE)
    .filter(time_filter)
    .filter(
        (F.col("InitiatingProcessFileName") == "outlook.exe") & 
        (F.col("ActionType") == "FileCreated")
    )
    .select(
        "TimeGenerated",
        "DeviceName",
        "SHA256"
    )
    .join(
        attachment_lookup,
        on="SHA256",
        how="inner"
    )
    .dropDuplicates(["SHA256", "DeviceName"])
)
df_device_files = cache_and_log_step("5. df_device_files (Attachments extracted by Outlook)", df_device_files, start)

## 6. Device Process Events: 
multi-signal approach. Signal 1 is for directly executed attachments.

In [ ]:
# Step 6: attachment Execution Detection (Multi-signal approach)
start = time.time()

# === 6.1: Direct SHA256 match (executable attachments) ===
# Works for: .exe, .dll, .scr, .com, .bat (files that ARE the process)
df_processes_sha256 = (
    lake_provider.read_table("DeviceProcessEvents", LOG_ANALYTICS_WORKSPACE)
    .filter(time_filter)
    .filter(F.col("SHA256").isNotNull())
    .select(
        "TimeGenerated",
        "DeviceName",
        "ProcessId",
        "InitiatingProcessId",
        "InitiatingProcessFileName",
        "InitiatingProcessCommandLine",
        "ProcessCommandLine",
        "FileName",
        F.col("SHA256").alias("ProcessSHA256")
    )
    .join(
        attachment_lookup.select(
            F.col("SHA256").alias("AttachmentSHA256")
        ),
        F.col("ProcessSHA256") == F.col("AttachmentSHA256"),
        how="inner"
    )
    .withColumn("MatchType", F.lit("SHA256_Direct"))
    .withColumn("ProcessKey", F.concat_ws("_", F.col("ProcessId"), F.col("DeviceName")))
)

df_processes_sha256 = cache_and_log_step("6a. df_processes_sha256 (SHA256 direct match)", df_processes_sha256, start)


In [ ]:
## Combine all detection signals

df_processes = df_processes_sha256 # for now, only one signal

## Rest of the notebook

In [ ]:
# Threat Intelligence
start = time.time()
df_ti = (
    lake_provider.read_table("ThreatIntelIndicators", LOG_ANALYTICS_WORKSPACE)
    .filter(ti_time_filter)
    .select(
        "ObservableKey",
        "ObservableValue",
        "Confidence",
        "Tags"
    )
)
df_ti = cache_and_log_step("7. df_ti (Threat Intelligence raw)", df_ti, start)
 
start = time.time()
df_ti_domains = (
    df_ti
    .filter(F.col("ObservableKey") == "domain-name:value")
    .select(
        F.col("ObservableValue").alias("Domain"),
        "Confidence",
        "Tags"
    )
)
df_ti_domains = cache_and_log_step("8. df_ti_domains (TI domains)", df_ti_domains, start)
 
start = time.time()
df_ti_urls = (
    df_ti
    .filter(F.col("ObservableKey") == "url:value")
    .select(
        F.col("ObservableValue").alias("Url"),
        "Confidence",
        "Tags"
    )
)
df_ti_urls = cache_and_log_step("9. df_ti_urls (TI URLs)", df_ti_urls, start)
print("=" * 80)

In [ ]:
from sentinel_graph.builders.graph_builder import GraphSpecBuilder
from sentinel_lake.providers import MicrosoftSentinelProvider
from pyspark.sql import functions as F
import time
 
print("=" * 80)
print("EMAIL AND ATTACHMENT ENRICHMENT PIPELINE")
print("=" * 80)
 
# Calculate sender-recipient communication frequency
start = time.time()
sender_recipient_freq = (
    df_emails
    .groupBy("SenderFromAddress", "RecipientEmailAddress")
    .agg(F.count("NetworkMessageId").alias("MessageCount"))
)
sender_recipient_freq = cache_and_log_step("1. sender_recipient_freq", sender_recipient_freq, start)

In [ ]:
# Calculate average message count per recipient (to identify outliers)
start = time.time()
recipient_stats = (
    df_emails
    .groupBy("RecipientEmailAddress")
    .agg(
        F.count(F.col("SenderFromAddress")).alias("TotalMessagesReceived"),
        F.countDistinct("SenderFromAddress").alias("UniqueSenders")
    )
)
recipient_stats = cache_and_log_step("2. recipient_stats", recipient_stats, start)
 
# Enrich emails with context: is sender common for this recipient?
start = time.time()
df_emails_enriched = (
    df_emails
    .join(sender_recipient_freq, on=["SenderFromAddress", "RecipientEmailAddress"], how="left")
    .join(recipient_stats, on="RecipientEmailAddress", how="left")
    .withColumn(
        "IsCommonSender",
        F.when(F.col("MessageCount") > 1, True).otherwise(False)
    )
    .withColumn(
        "SenderRarity",
        F.round(F.col("MessageCount") / F.col("TotalMessagesReceived") * 100, 2)
    )
)
df_emails_enriched = cache_and_log_step("3. df_emails_enriched (with sender context)", df_emails_enriched, start)
 
# Prepare keys for email, URL, attachment
start = time.time()
df_emails_enriched = df_emails_enriched.withColumn(
    "SentKey", F.concat_ws("_", F.col("SenderFromAddress"), F.col("NetworkMessageId"))
).withColumn(
    "ReceivedKey", F.concat_ws("_", F.col("RecipientEmailAddress"), F.col("NetworkMessageId"))
)
df_emails_enriched = cache_and_log_step("4. df_emails_enriched (with keys)", df_emails_enriched, start)
 
# Email Campaign/Cluster enrichment
start = time.time()
df_email_campaigns = (
    df_emails_enriched
    .groupBy("EmailClusterId")
    .agg(
        F.count("NetworkMessageId").alias("EmailCountInCluster"),
        F.countDistinct("SenderFromAddress").alias("UniqueSendersInCluster"),
        F.countDistinct("RecipientEmailAddress").alias("UniqueRecipientsInCluster"),
        F.min("TimeGenerated").alias("CampaignStartTime"),
        F.max("TimeGenerated").alias("CampaignEndTime"),
        F.first("Subject").alias("CommonSubject")
    )
)
df_email_campaigns = cache_and_log_step("5. df_email_campaigns (clustered)", df_email_campaigns, start)
 
# Enrich emails with campaign metrics
start = time.time()
df_emails_enriched = (
    df_emails_enriched
    .join(
        df_email_campaigns,
        on="EmailClusterId",
        how="left"
    )
    .withColumn(
        "CampaignScale",
        F.when(F.col("EmailCountInCluster") >= 100, "Large")
            .when(F.col("EmailCountInCluster") >= 20, "Medium")
            .when(F.col("EmailCountInCluster") >= 5, "Small")
            .otherwise("Isolated")
    )
)
df_emails_enriched = cache_and_log_step("6. df_emails_enriched (with campaign scale)", df_emails_enriched, start)
 
# Attachment reputation based on prevalence
start = time.time()
df_attachment_stats = (
    df_attachments
    .groupBy("SHA256")
    .agg(
        F.count("NetworkMessageId").alias("AttachmentFrequency"),
        F.countDistinct("NetworkMessageId").alias("UniqueEmailsWithAttachment"),
        F.min("TimeGenerated").alias("FirstSeenAttachment"),
        F.max("TimeGenerated").alias("LastSeenAttachment")
    )
)
df_attachment_stats = cache_and_log_step("7. df_attachment_stats (frequency analysis)", df_attachment_stats, start)
 
# Enrich attachments with stats
start = time.time()
df_attachments = (
    df_attachments
    .join(
        df_attachment_stats,
        on="SHA256",
        how="left"
    )
    .withColumn(
        "AttachmentRarity",
            F.when(F.col("AttachmentFrequency") == 1, "Unique")
            .when(F.col("AttachmentFrequency") <= 5, "Rare")
            .when(F.col("AttachmentFrequency") <= 20, "Uncommon")
            .otherwise("COMMON")
    )
    .withColumn(
        "EmailsWithAttachment",
        F.col("AttachmentFrequency")
    )
    .drop("AttachmentFrequency")
)
df_attachments = cache_and_log_step("8. df_attachments (enriched with rarity)", df_attachments, start)
 
print("=" * 80)
print("PREPARING NODE TABLES FOR GRAPH")
print("=" * 80)
 
# Prepare node tables
start = time.time()
sender_nodes = df_emails_enriched.select("SenderFromAddress").distinct()
sender_nodes = cache_and_log_step("9. sender_nodes", sender_nodes, start)
 
start = time.time()
email_nodes = df_emails_enriched.select("NetworkMessageId",
                                        "SenderFromAddress",
                                        "Subject",
                                        "EmailClusterId",
                                        "IsCommonSender",
                                        "MessageCount",
                                        "SenderRarity",
                                        "CampaignScale",
                                        "EmailCountInCluster",
                                        "UniqueRecipientsInCluster").distinct()
email_nodes = cache_and_log_step("10. email_nodes", email_nodes, start)
 
start = time.time()
user_nodes = df_emails_enriched.select("RecipientEmailAddress").distinct()
user_nodes = cache_and_log_step("11. user_nodes", user_nodes, start)
 
start = time.time()
url_nodes = df_urls.select("Url").distinct()
url_nodes = cache_and_log_step("12. url_nodes", url_nodes, start)
 
start = time.time()
domain_nodes = df_urls.select("Domain").distinct()
domain_nodes = cache_and_log_step("13. domain_nodes", domain_nodes, start)
 
start = time.time()
attachment_nodes = df_attachments.select("FileName", "SHA256", "AttachmentRarity", "EmailsWithAttachment").distinct()
attachment_nodes = cache_and_log_step("14. attachment_nodes", attachment_nodes, start)
 
start = time.time()
device_nodes = df_device_files.select("DeviceName").distinct()
device_nodes = cache_and_log_step("15. device_nodes", device_nodes, start)
 
start = time.time()
process_nodes = df_processes.select(
    "ProcessId",
    "ProcessKey",
    "InitiatingProcessFileName",
    "InitiatingProcessCommandLine",
    "ProcessCommandLine",
    "DeviceName"
).distinct()
process_nodes = cache_and_log_step("16. process_nodes", process_nodes, start)
 
start = time.time()
email_campaign_nodes = df_email_campaigns.select(
    "EmailClusterId",
    "EmailCountInCluster",
    "UniqueSendersInCluster",
    "UniqueRecipientsInCluster",
    "CampaignStartTime",
    "CampaignEndTime",
    "CommonSubject"
).distinct()
email_campaign_nodes = cache_and_log_step("17. email_campaign_nodes", email_campaign_nodes, start)
 
start = time.time()
ti_domain_nodes = df_ti_domains.select("Domain", "Confidence", "Tags").distinct()
ti_domain_nodes = cache_and_log_step("18. ti_domain_nodes", ti_domain_nodes, start)
 
start = time.time()
ti_url_nodes = df_ti_urls.select("Url", "Confidence", "Tags").distinct()
ti_url_nodes = cache_and_log_step("19. ti_url_nodes", ti_url_nodes, start)
 
print("=" * 80)
 
 

In [ ]:

# Build Graph
phishing_graph = (
    GraphSpecBuilder.start()
    
    # Sender
    .add_node("Sender")
    .from_dataframe(sender_nodes)
    .with_columns("SenderFromAddress", key="SenderFromAddress", display="SenderFromAddress")
    
    # Email
    .add_node("Email")
    .from_dataframe(email_nodes)
    .with_columns("NetworkMessageId",
                  "SenderFromAddress",
                  "Subject",
                  "EmailClusterId",
                  "IsCommonSender",
                  "MessageCount",
                  "SenderRarity",
                  "CampaignScale",
                  "EmailCountInCluster",
                  "UniqueRecipientsInCluster",
                  key="NetworkMessageId", display="NetworkMessageId")
    
    # User
    .add_node("User")
    .from_dataframe(user_nodes)
    .with_columns("RecipientEmailAddress", key="RecipientEmailAddress", display="RecipientEmailAddress")
    
    # URL (global -> One node per URL)
    .add_node("URL")
    .from_dataframe(url_nodes)
    .with_columns("Url", key="Url", display="Url")
    
    # Domain (global -> One node per URL)
    .add_node("Domain")
    .from_dataframe(domain_nodes)
    .with_columns("Domain", key="Domain", display="Domain")
    
    # Attachment
    .add_node("Attachment")
    .from_dataframe(attachment_nodes)
    .with_columns("FileName", "SHA256", "AttachmentRarity", key="SHA256", display="SHA256")
    
    # Device
    .add_node("Device")
    .from_dataframe(device_nodes)
    .with_columns("DeviceName", key="DeviceName", display="DeviceName")
 
    # Process
    .add_node("Process")
    .from_dataframe(process_nodes)
    .with_columns("ProcessId",
                  "ProcessKey",
                  "InitiatingProcessFileName",
                  "InitiatingProcessCommandLine",
                  "ProcessCommandLine",
                  "DeviceName",
                  key="ProcessKey", display="InitiatingProcessFileName")
 
    # EmailCluster/Campaign
    .add_node("EmailCampaign")
    .from_dataframe(email_campaign_nodes)
    .with_columns("EmailClusterId",
                  "EmailCountInCluster",
                  "UniqueSendersInCluster",
                  "UniqueRecipientsInCluster",
                  "CampaignStartTime",
                  "CampaignEndTime",
                  "CommonSubject",
                  key="EmailClusterId", display="EmailClusterId")
 
    # ThreatIntelligence
    .add_node("ThreatIndicator_Domain")
    .from_dataframe(ti_domain_nodes)
    .with_columns("Domain", "Confidence", "Tags", key="Domain", display="Domain")
 
    .add_node("ThreatIndicator_Url")
    .from_dataframe(ti_url_nodes)
    .with_columns("Url", "Confidence", "Tags", key="Url", display="Url")
 
    # Edges
    
    # Sender -> Email
    .add_edge("Sent")
    .from_dataframe(df_emails_enriched)
    .source(id_column="SenderFromAddress", node_type="Sender")
    .target(id_column="NetworkMessageId", node_type="Email")
    .with_columns("TimeGenerated", "SentKey", key="SentKey", display="SentKey")
    
    # User -> Email (Received)
    .add_edge("Received")
    .from_dataframe(df_emails_enriched)
    .source(id_column="RecipientEmailAddress", node_type="User")
    .target(id_column="NetworkMessageId", node_type="Email")
    .with_columns(
        "TimeGenerated",
        "ReceivedKey",
        "IsCommonSender",
        "MessageCount",
        "SenderRarity",
        key="ReceivedKey", display="ReceivedKey"
    )
 
    # Email -> URL
    .add_edge("HasUrl")
    .from_dataframe(df_urls)
    .source(id_column="NetworkMessageId", node_type="Email")
    .target(id_column="Url", node_type="URL")
    .with_columns("Url", key="Url", display="Url")
    
    # Email -> Domain
    .add_edge("HasDomain")
    .from_dataframe(df_urls)
    .source(id_column="NetworkMessageId", node_type="Email")
    .target(id_column="Domain", node_type="Domain")
    .with_columns("Domain", key="Domain", display="Domain")
 
    # Email -> Attachment
    .add_edge("HasAttachment")
    .from_dataframe(df_attachments)
    .source(id_column="NetworkMessageId", node_type="Email")
    .target(id_column="SHA256", node_type="Attachment")
    .with_columns("SHA256", key="SHA256", display="SHA256")
 
    # Attachment -> Device
    .add_edge("StoredOn")
    .from_dataframe(df_device_files)
    .source(id_column="SHA256", node_type="Attachment")
    .target(id_column="DeviceName", node_type="Device")
    .with_columns("DeviceName", key="DeviceName", display="DeviceName")
 
    # Attachment -> Process
    .add_edge("ExecutedBy")
    .from_dataframe(df_processes)
    .source(id_column="AttachmentSHA256", node_type="Attachment")
    .target(id_column="ProcessKey", node_type="Process")
    .with_columns("ProcessKey",
                  "ProcessId",
                  "FileName",
                  "InitiatingProcessFileName",
                  "InitiatingProcessCommandLine",
                  key="ProcessKey", display="InitiatingProcessCommandLine")
 
    # Domain -> TI_Domain
    .add_edge("DomainRelatedTo")
    .from_dataframe(df_ti_domains)
    .source(id_column="Domain", node_type="ThreatIndicator_Domain")
    .target(id_column="Domain", node_type="Domain")
    .with_columns("Domain", key="Domain", display="Domain")
 
    # URL -> TI_Url
    .add_edge("UrlRelatedTo")
    .from_dataframe(df_ti_urls)
    .source(id_column="Url", node_type="ThreatIndicator_Url")
    .target(id_column="Url", node_type="URL")
    .with_columns("Url", key="Url", display="Url")
 
    # Email -> EmailCampaign
    .add_edge("BelongsTo")
    .from_dataframe(df_emails_enriched)
    .source(id_column="NetworkMessageId", node_type="Email")
    .target(id_column="EmailClusterId", node_type="EmailCampaign")
    .with_columns("EmailClusterId", "CampaignScale", key="EmailClusterId", display="CampaignScale")
 
    # EmailCampaign -> User
    .add_edge("TargetsRecipients")
    .from_dataframe(df_emails_enriched.select("EmailClusterId", "RecipientEmailAddress").distinct())
    .source(id_column="EmailClusterId", node_type="EmailCampaign")
    .target(id_column="RecipientEmailAddress", node_type="User")
    .with_columns("RecipientEmailAddress", key="RecipientEmailAddress", display="RecipientEmailAddress")
 
).done()
 
# Step 4: Check schema
#phishing_graph.show_schema()

In [ ]:

build_result = phishing_graph.build_graph_with_data(provider="Fabric")
# print(f"Status: {build_result.get('status')}")
 


In [ ]:
# _query = """
# MATCH (s:Sender)-[sent:Sent]->(e:Email)
# MATCH (u:User)-[received:Received]->(e)
# OPTIONAL MATCH (e)-[hasUrl:HasUrl]->(url:URL)
# OPTIONAL MATCH (e)-[hasAtt:HasAttachment]->(att:Attachment)
# OPTIONAL MATCH (att)-[stored:StoredOn]->(d:Device)
# WHERE s.SenderFromAddress = 'stacey.crawford@vsoftcorp.com'
# RETURN
#   s, e, u, url, att, d,
#   sent, received, hasUrl, hasAtt, stored
# """
# phishing_graph.query(_query).show()
 
# _query = """
# MATCH (s:Sender)-[sent:Sent]->(e:Email)
# MATCH (u:User)-[received:Received]->(e)
# MATCH (e)-[hasAtt:HasAttachment]->(att:Attachment)
# MATCH (att)-[stored:StoredOn]->(d:Device)
# OPTIONAL MATCH (att)-[executed:ExecutedBy]->(p:Process)
# WHERE p IS NOT NULL
# RETURN
#   s.SenderFromAddress AS Sender,
#   e.NetworkMessageId AS Email,
#   u.RecipientEmailAddress AS User,
#   att.SHA256 AS AttachmentSHA256,
#   d.DeviceName AS Device,
#   p.ProcessKey AS ProcessKey,
#   p.ProcessCommandLine AS CommandLine,
#   stored, executed
# LIMIT 100
# """
# #  phishing_graph.query(_query).show()
 
# # Attachments with Process Execution
# _query = """
# MATCH (e:Email)-[hasAtt:HasAttachment]->(att:Attachment)
# MATCH (att)-[executed:ExecutedBy]->(p:Process)
# OPTIONAL MATCH (e)-[received:Received]->(u:User)
# OPTIONAL MATCH (e)-[belongsTo:BelongsTo]->(campaign:EmailCampaign)
# RETURN
#   att.FileName AS Filename,
#   att.SHA256 AS SHA256Hash,
#   att.AttachmentRarity AS AttachmentRarity,
#   e.NetworkMessageId AS EmailID,
#   e.Subject AS EmailSubject,
#   p.ProcessKey AS ProcessKey,
#   p.InitiatingProcessFileName AS InitiatingProcess,
#   p.ProcessCommandLine AS ProcessCommandLine,
#   p.DeviceName AS Device,
#   u.RecipientEmailAddress AS RecipientUser,
#   campaign.EmailClusterId AS CampaignID,
#   campaign.CampaignScale AS CampaignScale
# ORDER BY att.AttachmentRarity DESC
# LIMIT 100
# """